# Margin Console: where does a quick-commerce order lose money?

**Input:** `data/quick_comm2.csv` - the Kaggle *Quick Commerce* table (13 source columns) plus columns I engineered:
`Delivery_Cost`, `Discount_Amt`, `Profit`, `In_Loss`, `Loss_Driver`, and the bucket columns.
**Output:** cleaned tables in `data/` for the SQL/BigQuery pipeline and the Streamlit what-if app.

> **Read this first.** The source data has no cost fields. `Delivery_Cost` is a *modelled* cost, so every profit
> statement below is conditional on that model. Section 3 shows exactly what the model is. `Profit` here means
> *order value − delivery cost − discount*, i.e. **before product cost**, so it is best read as contribution
> before COGS, not net margin.

Set the path with the `QUICK_COMM_CSV` environment variable, or put the file at `data/quick_comm2.csv`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

DATA_PATH = Path(os.environ.get("QUICK_COMM_CSV", "../data/quick_comm2.csv"))
OUT_DIR = Path(os.environ.get("QUICK_COMM_OUT", "../data"))
print("reading", DATA_PATH.resolve())

## 1 · Load and clean (one dataframe, one pass)

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("raw shape:", raw.shape)

df = raw.loc[:, ~raw.columns.str.contains("^Unnamed")].copy()          # empty export columns
df = df.rename(columns={"Distance_time_bucket": "Delivery_Time_Bucket"})   # it is a time bucket
df["City"] = df["City"].str.strip().str.title().replace({"Bengluru": "Bengaluru"})
df["Company"] = df["Company"].str.strip()
df = df.astype({"Order_ID": "int64", "Customer_Age": "int64"})

print("clean shape:", df.shape)
df.info()

In [ ]:
# Quality checks. Each assert documents an invariant the rest of the notebook relies on.
assert df.isna().sum().sum() == 0, "nulls present"
assert not df["Order_ID"].duplicated().any(), "duplicate order ids"
assert not df.duplicated().any(), "duplicate rows"

positive = ["Order_Value", "Distance_Km", "Delivery_Time_Min", "Items_Count", "Delivery_Cost"]
assert (df[positive] > 0).all().all(), "non-positive values"
assert (df["Discount_Amt"] >= 0).all()
assert df["Customer_Rating"].between(1, 5).all() and df["Delivery_Partner_Rating"].between(1, 5).all()
assert set(df["Discount_Applied"].unique()) <= {0, 1} and set(df["In_Loss"].unique()) <= {0, 1}

for c in ["Company", "City", "Product_Category", "Payment_Method", "Distance_Bucket",
          "Age_Bucket", "Delivery_Time_Bucket"]:
    print(f"{c:22s}", sorted(df[c].unique()))

In [ ]:
# Profit reconciles to its components (rounding to 0.1 is the only allowed slack)
err = (df["Order_Value"] - df["Delivery_Cost"] - df["Discount_Amt"] - df["Profit"]).abs()
print("max reconciliation error:", err.max().round(4))
assert err.max() <= 0.11
assert ((df["Profit"] < 0).astype(int) == df["In_Loss"]).all(), "In_Loss disagrees with Profit < 0"

In [ ]:
# Outlier check on delivery time. Values are plausible (5-40 min), so they are KEPT, not trimmed.
q1, q3 = df["Delivery_Time_Min"].quantile([.25, .75])
fence = q3 + 1.5 * (q3 - q1)
print(f"IQR upper fence {fence:.0f} min | rows above: {(df['Delivery_Time_Min'] > fence).sum():,} "
      f"| max {df['Delivery_Time_Min'].max()} min")

## 2 · Features
Profit margin is computed two ways on purpose. The **per-order mean** is dragged down by tiny baskets (a ₹57 order
that loses ₹55 is −96%); the **revenue-weighted** margin (Σprofit ÷ Σorder value) is what a P&L would show.
(`Is_Weekend` from the old version is dropped: there is no date column, so it could only ever be empty.)

In [ ]:
df["Profit_Margin_pct"] = 100 * df["Profit"] / df["Order_Value"]
df["Cost_to_Serve_pct"] = 100 * (df["Delivery_Cost"] + df["Discount_Amt"]) / df["Order_Value"]
df["Basket_Band"] = pd.cut(df["Order_Value"], [0, 50, 100, 150, 200, 300, 600, 1000, np.inf],
                           labels=["<50", "50-99", "100-149", "150-199", "200-299", "300-599", "600-999", "1000+"],
                           right=False)
df[["Order_Value", "Delivery_Time_Min", "Distance_Km", "Delivery_Cost", "Profit"]].describe().round(2)

In [ ]:
axes = df[["Order_Value", "Delivery_Time_Min", "Distance_Km", "Profit"]].hist(figsize=(10, 7), bins=50)
for ax in axes.flatten():
    ax.set_xlabel(ax.get_title()); ax.set_ylabel("Orders"); ax.set_title("")
plt.tight_layout(); plt.show()

In [ ]:
for col in ["Company", "City", "Product_Category", "Payment_Method", "Age_Bucket"]:
    print((df[col].value_counts(normalize=True) * 100).round(1).to_string(), "\n")

**Read:** platforms, cities, categories and payment methods are each split almost exactly evenly
(≈12.5% per platform, ≈8.3% per city, ≈14.3% per category, ≈20% per payment method). Real marketplaces are never this
balanced, which suggests the source table is **simulated**. Findings about *which platform or city is worst* are therefore
a demonstration of method, not evidence about real companies.

## 3 · How is `Delivery_Cost` built?
Before interpreting any loss, check whether cost is measured or computed. If it is a straight line in distance, then
distance sets the bar a basket must clear and everything else follows from that.

In [ ]:
slope, intercept = np.polyfit(df["Distance_Km"], df["Delivery_Cost"], 1)
r2 = np.corrcoef(df["Distance_Km"], df["Delivery_Cost"])[0, 1] ** 2
print(f"Delivery_Cost = {intercept:.2f} + {slope:.2f} x km      R^2 = {r2:.6f}")
print(f"cost range: {df['Delivery_Cost'].min():.1f} to {df['Delivery_Cost'].max():.1f}")

s = df.sample(20_000, random_state=1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(s["Distance_Km"], s["Delivery_Cost"], s=3, alpha=.3)
ax[0].set(xlabel="Distance (km)", ylabel="Delivery cost (₹)", title="Cost vs distance")
km = np.linspace(0.5, 15, 50)
ax[1].plot(km, (intercept + slope * km) / (1 - df["Discount_Amt"].sum() / df["Order_Value"].sum()))
ax[1].set(xlabel="Distance (km)", ylabel="Break-even basket (₹)", title="Smallest basket that does not lose money")
plt.tight_layout(); plt.show()

**Read:** if R² is ≈ 1, cost is a *formula* of distance, not a measurement. It also means a basket at or above
the cost of the longest trip (plus its discount) can never lose money under this model. Break-even therefore rises with
distance, from about ₹45 at 2 km to about ₹149 at 15 km before discounts. The correlation of −0.09 between distance and profit
understates distance's role: it is diluted by the huge spread in basket size, but the *loss frontier* is entirely distance-driven.

## 4 · Who loses money?

In [ ]:
loss = df[df["In_Loss"] == 1]
print(f"loss-making orders: {len(loss):,} = {100 * df['In_Loss'].mean():.2f}% of {len(df):,}")
print(f"aggregate deficit : ₹{-loss['Profit'].sum():,.0f}    average loss per loss order: ₹{-loss['Profit'].mean():.2f}")
print(f"largest loss-making basket: ₹{loss['Order_Value'].max():,.0f}   |   smallest profitable basket at 15 km: "
      f"₹{(intercept + slope * 15):.0f}+")

In [ ]:
# The old finding "100% of losses are Low Value Orders" is true by construction, not by discovery.
print(pd.crosstab(loss["Loss_Driver"], "loss orders"))
print()
print("Share of ALL orders with basket < 300 (the label's first test):",
      f"{100 * (df['Order_Value'] < 300).mean():.1f}%")
print(f"Every loss basket is below ₹{loss['Order_Value'].max():,.0f}, so the < 300 test catches them all "
      "whatever order the other labels are tested in.")

**Read:** `Loss_Driver` was built with a `CASE` that tests *order value < 300 first*. Since no loss-making basket
exceeds that, 100% of losses get that label regardless of what else is true. It is a tautology, so it cannot be presented as
"the one true root cause". The honest, non-circular statement is the one in section 3: **a loss occurs when basket value is
below the trip's delivery cost (plus discount)**, and the table below shows how that frontier moves with distance.

In [ ]:
heat = (df.pivot_table(index="Basket_Band", columns="Distance_Bucket", values="In_Loss", aggfunc="mean",
                       observed=True) * 100).round(1)
plt.figure(figsize=(7, 4.5))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "% of orders losing money"})
plt.title("Loss rate by basket size and trip distance"); plt.ylabel("Basket (₹)"); plt.xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
num = ["Customer_Age", "Order_Value", "Delivery_Time_Min", "Distance_Km", "Items_Count",
       "Customer_Rating", "Delivery_Partner_Rating", "Delivery_Cost", "Discount_Amt", "Profit"]
plt.figure(figsize=(9, 7))
sns.heatmap(df[num].corr().round(2), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation matrix"); plt.tight_layout(); plt.show()

**Read:** two of the strongest correlations are *mechanical*, not insights: `Profit` is defined as order value minus
costs (so corr ≈ 1 with `Order_Value`), and `Delivery_Cost` is defined from `Distance_Km` (corr = 1). Nothing was learned
from either. The informative structure is in the heatmap above.

## 5 · Platforms and cities

In [ ]:
by_co = df.groupby("Company").agg(
    orders=("Order_ID", "count"),
    avg_basket=("Order_Value", "mean"),
    avg_profit=("Profit", "mean"),
    margin_per_order_mean=("Profit_Margin_pct", "mean"),
    margin_weighted=("Profit", lambda s: 100 * s.sum() / df.loc[s.index, "Order_Value"].sum()),
    loss_rate_pct=("In_Loss", lambda x: 100 * x.mean()),
).round(2).sort_values("margin_per_order_mean")
by_co

**Read:** the loss rate here is the share of *orders* with negative profit. It is **not** customer attrition (the old text
called it that; the data has no customer identity or repeat behaviour, so attrition cannot be measured). Jio Mart has the
lowest margin and highest loss rate, and also the smallest average basket - consistent with basket size being the driver.

In [ ]:
pivot = df.pivot_table(index="City", columns="Company", values="Profit", aggfunc="mean").round(1)
gap = pivot.sub(df.groupby("Company")["Profit"].mean(), axis=1)
plt.figure(figsize=(10, 5.5))
sns.heatmap(gap, annot=True, fmt=".0f", cmap="RdYlGn", center=0)
plt.title("Avg profit per order vs the platform's own average (₹)"); plt.tight_layout(); plt.show()

In [ ]:
# Why are some cities worse? Split average profit = basket - delivery cost - discount.
city = df.groupby("City").agg(avg_profit=("Profit", "mean"), avg_basket=("Order_Value", "mean"),
                              avg_delivery_cost=("Delivery_Cost", "mean"), avg_discount=("Discount_Amt", "mean"),
                              avg_km=("Distance_Km", "mean"), loss_rate_pct=("In_Loss", lambda x: 100 * x.mean()))
city = city.round(1).sort_values("avg_profit")
display(city)
best, worst = city.index[-1], city.index[0]
d = (city.loc[best] - city.loc[worst])
print(f"\n{best} vs {worst}: profit gap ₹{d['avg_profit']:.0f} = basket gap ₹{d['avg_basket']:.0f} "
      f"- cost gap ₹{d['avg_delivery_cost']:.0f} - discount gap ₹{d['avg_discount']:.0f}")

**Read:** the printed decomposition shows how much of the city gap comes from basket size versus trip distance
(cost). Whichever dominates is the real lever for that city. Because city effects repeat across every platform in the
heatmap, the explanation is market-level (basket and distance mix), not platform execution.

## 6 · Do discounts help profit?

In [ ]:
naive = df.groupby("Discount_Applied").agg(orders=("Order_ID", "count"), avg_basket=("Order_Value", "mean"),
                                           avg_discount=("Discount_Amt", "mean"), avg_profit=("Profit", "mean")).round(2)
display(naive)
print("profit ratio (discounted / not):", round(naive.loc[1, "avg_profit"] / naive.loc[0, "avg_profit"], 3))
print("basket ratio (discounted / not):", round(naive.loc[1, "avg_basket"] / naive.loc[0, "avg_basket"], 3))

# Discounted baskets are bigger in EVERY age group. This is the order-value uplift, not a profit uplift.
up = df.groupby(["Age_Bucket", "Discount_Applied"])["Order_Value"].mean().unstack()
up["basket_uplift_pct"] = (100 * (up[1] - up[0]) / up[0]).round(1)
display(up.round(1))

rate = (df.loc[df["Discount_Applied"] == 1, "Discount_Amt"] / df.loc[df["Discount_Applied"] == 1, "Order_Value"])
print("\nDiscount as a share of basket, discounted orders:")
print(rate.describe(percentiles=[.05, .5, .95]).round(4).to_string())

In [ ]:
# Like-for-like: compare inside basket-size bands so a bigger basket cannot pose as a discount effect
band = df.pivot_table(index="Basket_Band", columns="Discount_Applied", values=["Profit", "Order_Value"],
                      aggfunc="mean", observed=True).round(1)
band["profit_gap_(disc - no disc)"] = band[("Profit", 1)] - band[("Profit", 0)]
display(band)

**Read (how to interpret the three tables).** Discounted orders have a much larger average basket *and* higher average profit.
That is an **association**. Profit is `basket - delivery cost - discount`, so the discount itself is a cost on every discounted
order. The band table compares like with like: if the profit gap inside a basket band is small or negative, the headline
"59% more profitable" is mostly a **basket-size effect** (customers who get discounts order more), not a discount effect. If a gap
remains positive inside bands, it still cannot be called causal without randomisation - that is what the Experiment tab designs.
Also check the discount-rate summary: one visible row shows a 10% discount while the group average is about 1% of basket, so the
rule that generated `Discount_Amt` must be documented before making any discount-ROI claim.

## 7 · Is speed worth it?

In [ ]:
tb = df.groupby("Delivery_Time_Bucket").agg(orders=("Order_ID", "count"), avg_profit=("Profit", "mean"),
                                            avg_cost=("Delivery_Cost", "mean"), avg_km=("Distance_Km", "mean"),
                                            avg_rating=("Customer_Rating", "mean"),
                                            loss_rate_pct=("In_Loss", lambda x: 100 * x.mean())).round(2)
display(tb)
print("corr(delivery time, distance) =", df["Delivery_Time_Min"].corr(df["Distance_Km"]).round(3))

# Hold distance fixed: does time still matter?
within = df.pivot_table(index="Distance_Bucket", columns="Delivery_Time_Bucket", values="Profit",
                        aggfunc="mean", observed=True).round(1)
display(within)

**Read:** fast deliveries look far more profitable *and* travel a shorter distance, and cost is a function of distance. The
table above holds distance fixed. If profit differences shrink or vanish inside a distance bucket, "speed = profit" is really
"near = profit". Either way, this is an association between time and profit; it does not show that speeding up a given
delivery would add profit.

## 8 · Statistical checks

In [ ]:
a = df.loc[df["Discount_Applied"] == 1, "Profit"]
b = df.loc[df["Discount_Applied"] == 0, "Profit"]
t, p = stats.ttest_ind(a, b, equal_var=False)
diff = a.mean() - b.mean()
se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
pooled = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
print(f"Welch t = {t:.1f}, p = {p:.2g}")
print(f"difference in mean profit = ₹{diff:.1f}   95% CI [{diff - 1.96 * se:.1f}, {diff + 1.96 * se:.1f}]")
print(f"Cohen's d = {diff / pooled:.2f}   (n = {len(a):,} vs {len(b):,})")

**Read:** with ~950k rows almost any difference has p ≈ 0, so the p-value carries no information here. Report the *effect size* and
interval, and remember the test only says the two groups differ - not why (see section 6).

## 9 · Segments (K-means, exploratory)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

feat = df[["Order_Value", "Delivery_Time_Min", "Distance_Km", "Profit_Margin_pct"]].sample(150_000, random_state=42)
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(StandardScaler().fit_transform(feat))
feat = feat.assign(cluster=km.labels_)
display(feat.groupby("cluster").agg(orders=("Order_Value", "size"), basket=("Order_Value", "mean"),
                                    minutes=("Delivery_Time_Min", "mean"), km=("Distance_Km", "mean"),
                                    margin_pct=("Profit_Margin_pct", "mean")).round(1))

**Read:** one cluster is small baskets with a strongly negative margin - the same population the heatmap in section 4 isolates.
K-means is used descriptively (fit on a 150k sample for speed); it adds no evidence beyond the heatmap.

## 10 · Export for SQL, BigQuery and the app

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
out = df.drop(columns=["Profit_Margin_pct", "Cost_to_Serve_pct", "Basket_Band"]).copy()
out.columns = [c.lower() for c in out.columns]
out.to_parquet(OUT_DIR / "orders_clean.parquet", index=False)
out.to_csv(OUT_DIR / "orders_clean.csv", index=False)
slim = ["order_id", "company", "city", "product_category", "order_value", "distance_km",
        "delivery_time_min", "discount_applied", "delivery_cost", "discount_amt", "profit"]
out[slim].to_parquet(OUT_DIR / "orders.parquet", index=False)
print("wrote", [p.name for p in sorted(OUT_DIR.glob("orders*"))])

## Summary: what holds up, what changed

| Earlier claim | Status |
|---|---|
| 9.77% of orders are loss-making (92,592; ₹4.22M deficit) | **Holds**, under the modelled cost |
| "100% of losses trace to Low Value Orders = the one root cause" | **Reframed.** True by label construction; the real mechanism is basket < trip cost (sections 3-4) |
| "Distance doesn't independently cause losses" | **Withdrawn.** Cost is a function of distance; distance sets the break-even basket |
| "Discounting helps, 59% more profitable" | **Reframed.** Association driven by basket size; within a basket band a discount lowers profit (section 6) |
| "3× lower attrition" | **Relabelled.** It is loss-*order* rate; attrition is not measurable here |
| "Speed = profit" | **Qualified.** Confounded with distance (section 7) |
| "Corr(order value, profit) ≈ 1 = order value is king" | **Removed.** Mechanical by definition |
| Platform / city rankings | **Kept as demonstration**; data looks simulated |